In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

In [2]:
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

OUTPUT_DIR = Path("mmi711_outputs_multilabel")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Original event-presence dataset
data = np.load(OUTPUT_DIR / "mmi711_multilabel_variable_length_datasets.npz")

# New sequence-location labels
loc_data = np.load(OUTPUT_DIR / "mmi711_sequence_location_labels.npz")

with open(OUTPUT_DIR / "dataset_config_multilabel_variable_lengths.json", "r") as f:
    config = json.load(f)

LENGTHS = config["lengths"]
EVENT_LABELS = config["event_labels"]

print("Lengths:", LENGTHS)
print("Event labels:", EVENT_LABELS)
print("Location label arrays:", loc_data.files[:5], "...")

Device: cuda
Lengths: [400, 800, 1200]
Event labels: ['mean_shift', 'variance_shift', 'trend_shift', 'point_anomaly', 'collective_anomaly']
Location label arrays: ['Yloc_train_L400', 'Yloc_train_L800', 'Yloc_train_L1200', 'Yloc_val_L400', 'Yloc_val_L800'] ...


In [3]:
X_train_by_length = {}
Yloc_train_by_length = {}

X_val_by_length = {}
Yloc_val_by_length = {}

X_ood_params_by_length = {}
Yloc_ood_params_by_length = {}

X_ood_background_by_length = {}
Yloc_ood_background_by_length = {}

for length in LENGTHS:
    X_train_by_length[length] = data[f"X_train_L{length}"]
    Yloc_train_by_length[length] = loc_data[f"Yloc_train_L{length}"]

    X_val_by_length[length] = data[f"X_val_L{length}"]
    Yloc_val_by_length[length] = loc_data[f"Yloc_val_L{length}"]

    X_ood_params_by_length[length] = data[f"X_ood_params_L{length}"]
    Yloc_ood_params_by_length[length] = loc_data[f"Yloc_ood_params_L{length}"]

    X_ood_background_by_length[length] = data[f"X_ood_background_L{length}"]
    Yloc_ood_background_by_length[length] = loc_data[f"Yloc_ood_background_L{length}"]

    print(
        f"Length {length}:",
        "train", X_train_by_length[length].shape, Yloc_train_by_length[length].shape,
        "| val", X_val_by_length[length].shape, Yloc_val_by_length[length].shape,
        "| ood_params", X_ood_params_by_length[length].shape, Yloc_ood_params_by_length[length].shape,
        "| ood_background", X_ood_background_by_length[length].shape, Yloc_ood_background_by_length[length].shape
    )

Length 400: train (800, 400) (800, 400, 5) | val (240, 400) (240, 400, 5) | ood_params (240, 400) (240, 400, 5) | ood_background (240, 400) (240, 400, 5)
Length 800: train (800, 800) (800, 800, 5) | val (240, 800) (240, 800, 5) | ood_params (240, 800) (240, 800, 5) | ood_background (240, 800) (240, 800, 5)
Length 1200: train (800, 1200) (800, 1200, 5) | val (240, 1200) (240, 1200, 5) | ood_params (240, 1200) (240, 1200, 5) | ood_background (240, 1200) (240, 1200, 5)


In [4]:
def check_location_labels(Yloc_by_length, split_name):
    print("\n" + "=" * 70)
    print(split_name)
    print("=" * 70)

    Y_all = np.concatenate(
        [Yloc_by_length[length].reshape(-1, len(EVENT_LABELS)) for length in LENGTHS],
        axis=0
    )

    for i, label in enumerate(EVENT_LABELS):
        values, counts = np.unique(Y_all[:, i], return_counts=True)
        print(label, dict(zip(values.tolist(), counts.tolist())))

check_location_labels(Yloc_train_by_length, "Train")
check_location_labels(Yloc_val_by_length, "Validation")


Train
mean_shift {0: 1586976, 1: 281803, 2: 51221}
variance_shift {0: 1588512, 1: 284205, 2: 47283}
trend_shift {0: 1598427, 1: 272749, 2: 48824}
point_anomaly {0: 1919626, 1: 374}
collective_anomaly {0: 1864539, 1: 55461}

Validation
mean_shift {0: 472606, 1: 87927, 2: 15467}
variance_shift {0: 473180, 1: 86763, 2: 16057}
trend_shift {0: 478320, 1: 84475, 2: 13205}
point_anomaly {0: 575889, 1: 111}
collective_anomaly {0: 559516, 1: 16484}


In [5]:
class TimeSeriesLocationDataset(Dataset):
    def __init__(self, X, Yloc):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Yloc = torch.tensor(Yloc, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Yloc[idx]

In [6]:
class CNNBiLSTMLocationModel(nn.Module):
    def __init__(self, num_shift_types=3, num_anomaly_types=2, shift_classes=3, hidden_size=64, dropout=0.2):
        super().__init__()

        self.num_shift_types = num_shift_types
        self.num_anomaly_types = num_anomaly_types
        self.shift_classes = shift_classes

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.bilstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        feature_dim = hidden_size * 2

        self.dropout = nn.Dropout(dropout)

        # For mean/variance/trend shift:
        # output shape will be (batch, length, 3 shift types, shift_classes)
        self.shift_head = nn.Linear(feature_dim, num_shift_types * shift_classes)

        # For point/collective anomaly:
        # output shape will be (batch, length, 2 anomaly types)
        self.anomaly_head = nn.Linear(feature_dim, num_anomaly_types)

    def forward(self, x):
        # x: (batch, length)

        x = x.unsqueeze(1)              # (batch, 1, length)
        conv_features = self.cnn(x)     # (batch, 64, length)
        conv_features = conv_features.transpose(1, 2)  # (batch, length, 64)

        lstm_out, _ = self.bilstm(conv_features)       # (batch, length, hidden*2)
        features = self.dropout(lstm_out)

        shift_logits = self.shift_head(features)
        shift_logits = shift_logits.view(
            x.size(0),
            features.size(1),
            self.num_shift_types,
            self.shift_classes
        )

        anomaly_logits = self.anomaly_head(features)

        return shift_logits, anomaly_logits

In [7]:
BATCH_SIZE = 32

def create_location_loaders(X_by_length, Yloc_by_length, shuffle):
    loaders = []

    for length in LENGTHS:
        loader = DataLoader(
            TimeSeriesLocationDataset(
                X_by_length[length],
                Yloc_by_length[length]
            ),
            batch_size=BATCH_SIZE,
            shuffle=shuffle
        )
        loaders.append(loader)

    return loaders

train_loaders = create_location_loaders(X_train_by_length, Yloc_train_by_length, shuffle=True)
val_loaders = create_location_loaders(X_val_by_length, Yloc_val_by_length, shuffle=False)
ood_params_loaders = create_location_loaders(X_ood_params_by_length, Yloc_ood_params_by_length, shuffle=False)
ood_background_loaders = create_location_loaders(X_ood_background_by_length, Yloc_ood_background_by_length, shuffle=False)

print("Number of train loaders:", len(train_loaders))

Number of train loaders: 3


In [8]:
max_shift_label = 0

for length in LENGTHS:
    max_shift_label = max(
        max_shift_label,
        int(Yloc_train_by_length[length][:, :, :3].max()),
        int(Yloc_val_by_length[length][:, :, :3].max())
    )

SHIFT_CLASSES = max_shift_label + 1

print("Max shift label:", max_shift_label)
print("Number of shift classes:", SHIFT_CLASSES)

Max shift label: 2
Number of shift classes: 3


In [9]:
def compute_anomaly_pos_weight(Yloc_by_length):
    all_anom = []

    for length in LENGTHS:
        # point_anomaly and collective_anomaly columns
        all_anom.append(Yloc_by_length[length][:, :, 3:5].reshape(-1, 2))

    all_anom = np.concatenate(all_anom, axis=0)

    pos = all_anom.sum(axis=0)
    neg = len(all_anom) - pos

    pos_weight = neg / (pos + 1e-8)
    pos_weight = np.clip(pos_weight, 1.0, 50.0)

    return torch.tensor(pos_weight, dtype=torch.float32)


anomaly_pos_weight = compute_anomaly_pos_weight(Yloc_train_by_length).to(device)
print("Anomaly pos_weight:", anomaly_pos_weight)

Anomaly pos_weight: tensor([50.0000, 33.6189], device='cuda:0')


In [10]:
def compute_location_loss(
    shift_logits,
    anomaly_logits,
    Yloc,
    anomaly_pos_weight,
    shift_loss_weight=1.0,
    anomaly_loss_weight=1.0
):
    """
    shift_logits:   (batch, length, 3, shift_classes)
    anomaly_logits: (batch, length, 2)
    Yloc:           (batch, length, 5)
    """

    shift_targets = Yloc[:, :, :3].long()
    anomaly_targets = Yloc[:, :, 3:5].float()

    shift_losses = []

    for shift_idx in range(3):
        logits_i = shift_logits[:, :, shift_idx, :]      # (batch, length, classes)
        target_i = shift_targets[:, :, shift_idx]        # (batch, length)

        loss_i = F.cross_entropy(
            logits_i.reshape(-1, logits_i.size(-1)),
            target_i.reshape(-1)
        )
        shift_losses.append(loss_i)

    shift_loss = torch.stack(shift_losses).mean()

    anomaly_loss = F.binary_cross_entropy_with_logits(
        anomaly_logits,
        anomaly_targets,
        pos_weight=anomaly_pos_weight
    )

    total_loss = shift_loss_weight * shift_loss + anomaly_loss_weight * anomaly_loss

    return total_loss, shift_loss.detach(), anomaly_loss.detach()

In [11]:
def apply_location_predictions(shift_logits, anomaly_logits, anomaly_threshold=0.5):
    shift_pred = torch.argmax(shift_logits, dim=-1)  # (batch, length, 3)

    anomaly_prob = torch.sigmoid(anomaly_logits)
    anomaly_pred = (anomaly_prob >= anomaly_threshold).long()  # (batch, length, 2)

    pred_loc = torch.cat([shift_pred, anomaly_pred], dim=-1)

    return pred_loc, anomaly_prob


def compute_pointwise_event_metrics_from_flat(y_true_flat, y_pred_flat):
    """
    y_true_flat and y_pred_flat shape:
        (total_time_points, 5)

    Shift columns may contain regime labels 0,1,2,...
    For pointwise event detection metrics, shifts are converted to binary:
        active shift = label > 0
    """
    y_true_binary = y_true_flat.copy()
    y_pred_binary = y_pred_flat.copy()

    # Shift columns: active if regime label > 0
    y_true_binary[:, :3] = (y_true_binary[:, :3] > 0).astype(int)
    y_pred_binary[:, :3] = (y_pred_binary[:, :3] > 0).astype(int)

    return {
        "pointwise_macro_f1": f1_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_micro_f1": f1_score(
            y_true_binary,
            y_pred_binary,
            average="micro",
            zero_division=0
        ),
        "pointwise_macro_precision": precision_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_macro_recall": recall_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_hamming_loss": hamming_loss(
            y_true_binary,
            y_pred_binary
        )
    }


def compute_shift_regime_accuracy_from_flat(y_true_flat, y_pred_flat):
    rows = {}

    for i, label in enumerate(EVENT_LABELS[:3]):
        rows[f"{label}_regime_accuracy"] = accuracy_score(
            y_true_flat[:, i],
            y_pred_flat[:, i]
        )

    return rows



In [12]:
def run_one_epoch_location(model, loaders, optimizer=None, anomaly_threshold=0.5):
    is_train = optimizer is not None

    if is_train:
        model.train()
        loaders = loaders.copy()
        random.shuffle(loaders)
    else:
        model.eval()

    total_loss = 0.0
    total_shift_loss = 0.0
    total_anomaly_loss = 0.0
    total_samples = 0

    all_true_flat = []
    all_pred_flat = []

    for loader in loaders:
        for X_batch, Yloc_batch in loader:
            X_batch = X_batch.to(device)
            Yloc_batch = Yloc_batch.to(device)

            if is_train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(is_train):
                shift_logits, anomaly_logits = model(X_batch)

                loss, shift_loss, anomaly_loss = compute_location_loss(
                    shift_logits=shift_logits,
                    anomaly_logits=anomaly_logits,
                    Yloc=Yloc_batch,
                    anomaly_pos_weight=anomaly_pos_weight
                )

                if is_train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            batch_size = X_batch.size(0)

            total_loss += loss.item() * batch_size
            total_shift_loss += shift_loss.item() * batch_size
            total_anomaly_loss += anomaly_loss.item() * batch_size
            total_samples += batch_size

            pred_loc, _ = apply_location_predictions(
                shift_logits,
                anomaly_logits,
                anomaly_threshold=anomaly_threshold
            )

            # Flatten each batch from (batch, length, 5) to (batch*length, 5)
            # This avoids errors when different loaders have different lengths.
            true_flat = Yloc_batch.detach().cpu().numpy().reshape(-1, len(EVENT_LABELS))
            pred_flat = pred_loc.detach().cpu().numpy().reshape(-1, len(EVENT_LABELS))

            all_true_flat.append(true_flat)
            all_pred_flat.append(pred_flat)

    y_true_flat = np.concatenate(all_true_flat, axis=0)
    y_pred_flat = np.concatenate(all_pred_flat, axis=0)

    metrics = compute_pointwise_event_metrics_from_flat(
        y_true_flat,
        y_pred_flat
    )

    metrics.update(
        compute_shift_regime_accuracy_from_flat(
            y_true_flat,
            y_pred_flat
        )
    )

    return {
        "loss": total_loss / total_samples,
        "shift_loss": total_shift_loss / total_samples,
        "anomaly_loss": total_anomaly_loss / total_samples,
        **metrics
    }

In [13]:
model = CNNBiLSTMLocationModel(
    shift_classes=SHIFT_CLASSES,
    hidden_size=64,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 100
ANOMALY_THRESHOLD = 0.5

history = []
best_val_f1 = 0.0
best_model_path = OUTPUT_DIR / "best_location_cnn_bilstm_model.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch_location(
        model,
        train_loaders,
        optimizer=optimizer,
        anomaly_threshold=ANOMALY_THRESHOLD
    )

    val_metrics = run_one_epoch_location(
        model,
        val_loaders,
        optimizer=None,
        anomaly_threshold=ANOMALY_THRESHOLD
    )

    row = {"epoch": epoch}

    for key, value in train_metrics.items():
        row[f"train_{key}"] = value

    for key, value in val_metrics.items():
        row[f"val_{key}"] = value

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Pointwise Macro F1: {train_metrics['pointwise_macro_f1']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Pointwise Macro F1: {val_metrics['pointwise_macro_f1']:.4f}"
    )

    if val_metrics["pointwise_macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["pointwise_macro_f1"]

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "event_labels": EVENT_LABELS,
                "lengths": LENGTHS,
                "shift_classes": SHIFT_CLASSES,
                "epoch": epoch,
                "best_val_pointwise_macro_f1": best_val_f1,
                "anomaly_threshold": ANOMALY_THRESHOLD
            },
            best_model_path
        )

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history_location_model.csv", index=False)

print("Training finished.")
print("Best validation pointwise macro F1:", best_val_f1)
print("Best model saved to:", best_model_path)

Epoch 01 | Train Loss: 1.0518 | Train Pointwise Macro F1: 0.0926 | Val Loss: 0.8275 | Val Pointwise Macro F1: 0.0927
Epoch 02 | Train Loss: 0.8058 | Train Pointwise Macro F1: 0.0979 | Val Loss: 0.8193 | Val Pointwise Macro F1: 0.0635
Epoch 03 | Train Loss: 0.7658 | Train Pointwise Macro F1: 0.1226 | Val Loss: 0.7609 | Val Pointwise Macro F1: 0.1283
Epoch 04 | Train Loss: 0.7236 | Train Pointwise Macro F1: 0.1620 | Val Loss: 0.7056 | Val Pointwise Macro F1: 0.1629
Epoch 05 | Train Loss: 0.6893 | Train Pointwise Macro F1: 0.1717 | Val Loss: 0.6796 | Val Pointwise Macro F1: 0.1515
Epoch 06 | Train Loss: 0.6706 | Train Pointwise Macro F1: 0.1834 | Val Loss: 0.6519 | Val Pointwise Macro F1: 0.1696
Epoch 07 | Train Loss: 0.6514 | Train Pointwise Macro F1: 0.2002 | Val Loss: 0.6698 | Val Pointwise Macro F1: 0.1615
Epoch 08 | Train Loss: 0.6554 | Train Pointwise Macro F1: 0.1906 | Val Loss: 0.6469 | Val Pointwise Macro F1: 0.1765
Epoch 09 | Train Loss: 0.6442 | Train Pointwise Macro F1: 0.2058

In [14]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded best model from epoch:", checkpoint["epoch"])
print("Best val pointwise macro F1:", checkpoint["best_val_pointwise_macro_f1"])

Loaded best model from epoch: 94
Best val pointwise macro F1: 0.6596337815640908


In [15]:
val_metrics = run_one_epoch_location(
    model,
    val_loaders,
    optimizer=None,
    anomaly_threshold=ANOMALY_THRESHOLD
)

ood_params_metrics = run_one_epoch_location(
    model,
    ood_params_loaders,
    optimizer=None,
    anomaly_threshold=ANOMALY_THRESHOLD
)

ood_background_metrics = run_one_epoch_location(
    model,
    ood_background_loaders,
    optimizer=None,
    anomaly_threshold=ANOMALY_THRESHOLD
)

location_results_df = pd.DataFrame([
    {"dataset": "validation", **val_metrics},
    {"dataset": "ood_params", **ood_params_metrics},
    {"dataset": "ood_background", **ood_background_metrics}
])

location_results_df.to_csv(OUTPUT_DIR / "final_results_location_model.csv", index=False)
location_results_df

,dataset,loss,shift_loss,anomaly_loss,pointwise_macro_f1,pointwise_micro_f1,pointwise_macro_precision,pointwise_macro_recall,pointwise_hamming_loss,mean_shift_regime_accuracy,variance_shift_regime_accuracy,trend_shift_regime_accuracy
0,validation,0.536289,0.327265,0.209025,0.659634,0.727016,0.686811,0.731844,0.054859,0.905661,0.900818,0.874568
1,ood_params,0.687594,0.486301,0.201293,0.524997,0.562707,0.602152,0.484403,0.082107,0.869648,0.859674,0.819967
2,ood_background,1.141484,0.590157,0.551326,0.493845,0.471954,0.497177,0.601672,0.112858,0.745089,0.829179,0.826295


In [16]:
def extract_persistent_shift_starts(seq, min_persistence):
    """
    Extracts shift start indices from a predicted regime sequence.

    A shift start is accepted only if the new active regime persists
    for at least min_persistence time steps.

    seq example:
        0 0 0 1 1 1 1 ...
    """
    seq = np.asarray(seq)

    starts = []

    # Find upward transitions: 0->1, 1->2, etc.
    diff = np.diff(seq, prepend=seq[0])
    candidate_starts = np.where(diff > 0)[0]

    for start in candidate_starts:
        new_label = seq[start]

        if new_label <= 0:
            continue

        end = start

        while end < len(seq) and seq[end] >= new_label:
            end += 1

        duration = end - start

        if duration >= min_persistence:
            starts.append(start)

    return starts

In [17]:
def extract_persistent_shift_starts(seq, min_persistence):
    """
    Extract shift start indices from a predicted or true shift-regime sequence.

    A shift start is accepted only if the active post-shift region
    continues for at least min_persistence time steps.
    """
    seq = np.asarray(seq)

    starts = []

    diff = np.diff(seq, prepend=0)
    candidate_starts = np.where(diff > 0)[0]

    for start in candidate_starts:
        new_label = seq[start]

        if new_label <= 0:
            continue

        end = start

        while end < len(seq) and seq[end] >= new_label:
            end += 1

        duration = end - start

        if duration >= min_persistence:
            starts.append(start)

    return starts


def extract_starts_from_label_sequence(seq, is_shift, min_persistence=None):
    seq = np.asarray(seq)

    if is_shift:
        if min_persistence is None:
            min_persistence = max(3, int(0.05 * len(seq)))

        starts = extract_persistent_shift_starts(
            seq,
            min_persistence=min_persistence
        )

    else:
        binary = (seq > 0).astype(int)
        diff = np.diff(binary, prepend=0)
        starts = np.where(diff == 1)[0]

    return starts


def match_start_locations(true_starts, pred_starts, tolerance):
    true_starts = list(true_starts)
    pred_starts = list(pred_starts)

    used_pred = set()
    abs_errors = []
    tp = 0

    for true_start in true_starts:
        best_j = None
        best_error = None

        for j, pred_start in enumerate(pred_starts):
            if j in used_pred:
                continue

            error = abs(pred_start - true_start)

            if best_error is None or error < best_error:
                best_error = error
                best_j = j

        if best_error is not None and best_error <= tolerance:
            tp += 1
            used_pred.add(best_j)
            abs_errors.append(best_error)

    fp = len(pred_starts) - len(used_pred)
    fn = len(true_starts) - tp

    return tp, fp, fn, abs_errors

In [18]:
def get_location_predictions_by_length(model, loaders, lengths, anomaly_threshold=0.5):
    """
    Returns predictions separately for each sequence length.

    Output:
        predictions_by_length[length] = {
            "y_true": array with shape (num_samples, length, 5),
            "y_pred": array with shape (num_samples, length, 5)
        }
    """
    model.eval()

    predictions_by_length = {}

    with torch.no_grad():
        for length, loader in zip(lengths, loaders):
            all_true = []
            all_pred = []

            for X_batch, Yloc_batch in loader:
                X_batch = X_batch.to(device)

                shift_logits, anomaly_logits = model(X_batch)

                pred_loc, _ = apply_location_predictions(
                    shift_logits,
                    anomaly_logits,
                    anomaly_threshold=anomaly_threshold
                )

                all_true.append(Yloc_batch.numpy())
                all_pred.append(pred_loc.cpu().numpy())

            predictions_by_length[length] = {
                "y_true": np.concatenate(all_true, axis=0),
                "y_pred": np.concatenate(all_pred, axis=0)
            }

    return predictions_by_length

In [19]:
def evaluate_event_start_locations_by_length(
    predictions_by_length,
    event_labels,
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.05
):
    """
    Evaluates event-start localization separately for each sequence length.

    For shifts, predicted starts are accepted only if the predicted
    post-shift region persists for at least shift_persistence_ratio
    of the sequence length.

    TN definition:
        sample-level true negative for a given event type:
        no true start exists and no predicted start exists.
    """
    rows = []

    for length, pred_dict in predictions_by_length.items():
        y_true_loc = pred_dict["y_true"]
        y_pred_loc = pred_dict["y_pred"]

        n_samples, seq_len, n_events = y_true_loc.shape

        tolerance = max(3, int(seq_len * tolerance_ratio))
        min_persistence = max(3, int(seq_len * shift_persistence_ratio))

        for event_idx, label in enumerate(event_labels):
            is_shift = event_idx < 3

            total_tp = 0
            total_fp = 0
            total_fn = 0
            total_tn = 0
            all_errors = []

            for sample_idx in range(n_samples):
                true_seq = y_true_loc[sample_idx, :, event_idx]
                pred_seq = y_pred_loc[sample_idx, :, event_idx]

                true_starts = extract_starts_from_label_sequence(
                    true_seq,
                    is_shift=is_shift,
                    min_persistence=1 if is_shift else None
)

                pred_starts = extract_starts_from_label_sequence(
                    pred_seq,
                    is_shift=is_shift,
                    min_persistence=min_persistence if is_shift else None
                )

                # Sample-level TN: no real start and no predicted start.
                if len(true_starts) == 0 and len(pred_starts) == 0:
                    total_tn += 1
                    continue

                tp, fp, fn, errors = match_start_locations(
                    true_starts=true_starts,
                    pred_starts=pred_starts,
                    tolerance=tolerance
                )

                total_tp += tp
                total_fp += fp
                total_fn += fn
                all_errors.extend(errors)

            precision = total_tp / (total_tp + total_fp + 1e-8)
            recall = total_tp / (total_tp + total_fn + 1e-8)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)

            specificity = total_tn / (total_tn + total_fp + 1e-8)
            balanced_accuracy = 0.5 * (recall + specificity)

            mean_abs_error = np.mean(all_errors) if len(all_errors) > 0 else np.nan

            rows.append({
                "length": length,
                "event": label,
                "tolerance_points": tolerance,
                "shift_min_persistence": min_persistence if is_shift else np.nan,
                "tp": total_tp,
                "fp": total_fp,
                "fn": total_fn,
                "tn": total_tn,
                "start_precision": precision,
                "start_recall": recall,
                "start_f1": f1,
                "start_specificity": specificity,
                "start_balanced_accuracy": balanced_accuracy,
                "mean_abs_start_error": mean_abs_error
            })

    return pd.DataFrame(rows)

In [20]:
val_predictions_by_length = get_location_predictions_by_length(
    model=model,
    loaders=val_loaders,
    lengths=LENGTHS,
    anomaly_threshold=ANOMALY_THRESHOLD
)

start_eval_val = evaluate_event_start_locations_by_length(
    predictions_by_length=val_predictions_by_length,
    event_labels=EVENT_LABELS,
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.1
)

start_eval_val

,length,event,tolerance_points,shift_min_persistence,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error
0,400,mean_shift,20,40.0,46,38,50,147,0.547619,0.479167,0.511111,0.794595,0.636881,6.260870
1,400,variance_shift,20,40.0,35,38,65,155,0.479452,0.350000,0.404624,0.803109,0.576554,7.314286
2,400,trend_shift,20,40.0,19,38,73,155,0.333333,0.206522,0.255034,0.803109,0.504815,8.842105
3,400,point_anomaly,20,NaN,26,79,9,156,0.247619,0.742857,0.371429,0.663830,0.703343,0.000000
4,400,collective_anomaly,20,NaN,64,28,11,150,0.695652,0.853333,0.766467,0.842697,0.848015,2.359375
5,800,mean_shift,40,80.0,51,32,40,154,0.614458,0.560440,0.586207,0.827957,0.694198,5.941176
6,800,variance_shift,40,80.0,43,38,52,156,0.530864,0.452632,0.488636,0.804124,0.628378,13.697674
7,800,trend_shift,40,80.0,15,41,77,156,0.267857,0.163043,0.202703,0.791878,0.477461,19.933333
8,800,point_anomaly,40,NaN,30,122,6,153,0.197368,0.833333,0.319149,0.556364,0.694848,0.833333
9,800,collective_anomaly,40,NaN,72,50,3,139,0.590164,0.960000,0.730964,0.735450,0.847725,2.763889
